# Batch prompt upsampling

Reads a prompts file (one `name: short prompt` per line, e.g. `seamless_prompts.txt`), runs every prompt through the
prompt upsampler (text-to-image mode, no references, no boxes) and writes `name: upsampled prompt` lines to the output
file, which `seamless_batch.ipynb` consumes. The output is rewritten after every prompt, so a crash loses nothing.

## Pick a GPU

Nothing runs until `GPU` is set. The index is the one `nvidia-smi` prints; it is exported as `CUDA_VISIBLE_DEVICES`
*before* torch is imported, so inside the notebook the chosen card is always `cuda:0` (vLLM has no device argument and
only honours this variable). Restart the kernel to switch.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

GPU = None  # <-- set me, e.g. GPU = 0

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert GPU is not None, "set GPU to one of the indices above before running anything else"
assert "torch" not in sys.modules, "torch is already imported: restart the kernel, then run this cell first"
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
DEVICE = "cuda"  # == cuda:0 == physical GPU above

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
os.environ.setdefault("WEIGHTS_DIR", str(REPO / "weights"))  # flux3 reads this at import time
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))


## Paths and model

In [ ]:
from flux3.pup import MODEL, Upsampler
from seamless import read_prompts, write_prompts

PROMPTS_IN = REPO / "seamless_prompts.txt"
PROMPTS_OUT = REPO / "notebooks" / "outputs" / "seamless_prompts_upsampled.txt"
MEMORY = 0.85  # vLLM's share of the GPU

prompts = read_prompts(PROMPTS_IN)
print(f"{len(prompts)} prompts from {PROMPTS_IN}")
pup = Upsampler(MODEL, MEMORY)

## Upsample

In [ ]:
PROMPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
upsampled = read_prompts(PROMPTS_OUT) if PROMPTS_OUT.exists() else {}  # resume: names already done are skipped

for name, prompt in prompts.items():
    if name in upsampled:
        print(f"[{name}] already done, skipping")
        continue
    upsampled[name] = pup(prompt, [], None)
    write_prompts(PROMPTS_OUT, {n: upsampled[n] for n in prompts if n in upsampled})
    print(f"[{name}]\n  in : {prompt}\n  out: {upsampled[name]}\n")
print(f"wrote {len(upsampled)} prompts to {PROMPTS_OUT}")